In [1]:
import os
import sys
import json

import matplotlib
matplotlib.use("module://ipympl.backend_nbagg")  # interaktives Backend
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, clear_output

import pandas as pd
import pyarrow.dataset as ds
import pyarrow.fs as pafs

# scenario_plot.py liegt im Repo-Root — Pfad hinzufügen
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
from scenario_plot import plot_hit

AWS_REGION  = "eu-central-1"
RESULTS_BUCKET = "womd-features"
FEATURES_PREFIX = "parquet/run-001"

In [2]:
s3 = pafs.S3FileSystem(region=AWS_REGION)

hits = ds.dataset(
    f"{RESULTS_BUCKET}/results/match_hits",
    filesystem=s3,
    format="parquet",
    partitioning="hive",
).to_table().to_pandas()

print(f"Hits geladen: {len(hits)}")
print(hits.groupby("scenario")["scene_id"].count().rename("n_hits"))

Hits geladen: 1155
scenario
ccrb.osc           508
change_lane.osc    534
cross.osc          113
Name: n_hits, dtype: int64


In [ ]:
# ── Filter-Controls ───────────────────────────────────────────────────────────
scenario_dd = widgets.Dropdown(
    options=sorted(hits["scenario"].unique().tolist()),
    value=hits["scenario"].iloc[0],
    description="Scenario:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="350px"),
)

min_windows_slider = widgets.IntSlider(
    value=1, min=1, max=int(hits["n_windows"].max()),
    step=1,
    description="Min windows:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="400px"),
)

shard_dd = widgets.Dropdown(
    options=["alle"] + sorted(hits["shard_index"].unique().tolist()),
    value="alle",
    description="Shard:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="200px"),
)

# ── Hit-Auswahl ───────────────────────────────────────────────────────────────
hit_dd = widgets.Dropdown(
    description="Hit:",
    style={"description_width": "40px"},
    layout=widgets.Layout(width="800px"),
)

hit_count_label = widgets.Label(value="")

# ── Plot-Output ───────────────────────────────────────────────────────────────
plot_out = widgets.Output(
    layout=widgets.Layout(width="100%", min_height="500px")
)

# ── State ─────────────────────────────────────────────────────────────────────
_filtered = pd.DataFrame()


def _make_label(i, row) -> str:
    roles = json.loads(row["roles_json"])
    roles_str = "  ".join(f"{k}={v}" for k, v in roles.items())
    return (
        f"#{i:04d}  |  scene {row['scene_id'][:12]}  |  {row['segment_id']}  |  "
        f"t0={row['t0']} → t1={row['t1']}  |  "
        f"n_win={row['n_windows']}  |  {roles_str}"
    )


def update_hit_list(*args):
    global _filtered

    mask = hits["scenario"] == scenario_dd.value
    mask &= hits["n_windows"] >= min_windows_slider.value
    if shard_dd.value != "alle":
        mask &= hits["shard_index"] == int(shard_dd.value)

    _filtered = hits[mask].reset_index(drop=True)

    options = [
        (_make_label(i, row), i)
        for i, row in _filtered.iterrows()
    ]
    hit_dd.options = options
    hit_count_label.value = f"{len(_filtered)} Hits gefunden"

    if options:
        hit_dd.value = options[0][1]


def on_hit_selected(change):
    if change["type"] != "change" or change["name"] != "value":
        return
    if _filtered.empty or change["new"] is None:
        return

    hit = _filtered.iloc[change["new"]]

    with plot_out:
        clear_output(wait=True)
        fig = plot_hit(
            hit,
            scenes_dir=f"s3://{RESULTS_BUCKET}/{FEATURES_PREFIX}",
            show_road=True,
            show_polygons=True,
            show_reference_line=True,
            show_trajectories=True,
            show_interaction=True,
            show_markers=True,
            n_arrows=6,
            figsize=(14, 9),
        )
        plt.show()


# ── Observers ─────────────────────────────────────────────────────────────────
scenario_dd.observe(update_hit_list, names="value")
min_windows_slider.observe(update_hit_list, names="value")
shard_dd.observe(update_hit_list, names="value")
hit_dd.observe(on_hit_selected, names="value")

# ── Layout ────────────────────────────────────────────────────────────────────
controls = widgets.VBox([
    widgets.HBox([scenario_dd, shard_dd]),
    widgets.HBox([min_windows_slider, hit_count_label]),
    hit_dd,
])

display(widgets.VBox([controls, plot_out]))

# Initial laden
update_hit_list()